# **LOGISTIC REGRESSION + EMBEDDINGS**

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report, f1_score, accuracy_score, precision_recall_fscore_support)
import json
import joblib
from pathlib import Path
import mlflow
import time
from sentence_transformers import SentenceTransformer

In [2]:
# get class to index 
df = pd.read_csv("../datasets/train.csv")
class_to_idx = {label:idx for idx, label in enumerate(sorted(df['labels'].unique()))}

In [16]:
# clean text

def clean_text_for_embeddings(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"([!\"'#$%&()*\+-/:;<=>?@\\\[\]^_`{|}~])", r" \1 ", sentence)
    sentence = re.sub("[^A-Za-z0-9]+", " ", sentence)
    pattern = re.compile(r"(xx+\s*)+")
    sentence = pattern.sub("mask ", sentence)
    sentence = re.sub(" +", " ", sentence).strip()
    sentence = " ".join(sentence.split())
    sentence = re.sub(r"http\S+", "", sentence)
    return sentence

In [4]:
train_path = "../datasets/train_embeddings.npy"
val_path = "../datasets/val_embeddings.npy"

with open(train_path, "rb") as f:
  X_train_embeddings = np.load(f)
  y_train = np.load(f)

with open(val_path, "rb") as f:
  X_val_embeddings = np.load(f)  
  y_val = np.load(f)

### **Trainer model class**

In [5]:
class EmbeddingsClassifier:
    def __init__(
            self, 
            model: LogisticRegression, 
            directory:str,
            train_dataset:tuple[np.array, np.array],
            val_dataset:tuple[np.array, np.array],
            experiment_name=None,
            log_experiment:bool=False,
            params=None,
            ):
        if params==None:
            self.params = None
            self.model = model()
        else:
            self.params = params
            self.model = model(**self.params) # i will save this

        self.directory_name = directory # path i will use to save above
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.experiment_name = experiment_name
        self.log_experiment = log_experiment

    def save_artifacts(self, model):
        artifacts = {"model": model}
        artifacts_path = Path().cwd().parent / self.directory_name
        if not artifacts_path.exists():
            artifacts_path.mkdir(parents=True)
        joblib.dump(artifacts, artifacts_path / "embeddings_model.joblib")

    def mlflow_logging(self, train_embeddings, train_labels):
        mlflow.set_tracking_uri("http://127.0.0.1:5000")
        mlflow.set_experiment(self.experiment_name)
        
        with mlflow.start_run():
            self.model.fit(train_embeddings, train_labels)
            y_train_pred = self.model.predict(train_embeddings)

            X_val, y_val = self.val_dataset[0], self.val_dataset[1]
            y_val_pred = self.model.predict(X_val)

            for name, y_true, y_pred in [("train", train_labels, y_train_pred), 
                                         ("val", y_val, y_val_pred)]:

                metrics = precision_recall_fscore_support(y_true=y_true, y_pred=y_pred, average='macro')
                mlflow.log_metrics({f"acc_{name}": accuracy_score(y_true=y_true, y_pred=y_pred), 
                                    f"prec_macro_{name}": metrics[0],
                                    f"recall_macro_{name}": metrics[1],
                                    f"f1_macro_{name}": metrics[2],
                                    f"weighted_f1_{name}": f1_score(y_true=y_true, y_pred=y_pred, average='weighted')
                                    })
            if self.params != None:
                mlflow.log_params(self.params)     

    def train(self):
        start = time.time()
        X_train, y_train= self.train_dataset[0], self.train_dataset[1]
        print(f"Train Examples: {X_train.shape[0]}")

        if self.log_experiment:
            print(f"Training and Logging to MLFLOW experiment {self.experiment_name}...")
            self.mlflow_logging(train_embeddings=X_train, train_labels=y_train)
        else:
            print("Training...")
            self.model.fit(X_train, y_train)
        
        self.save_artifacts(model=self.model)
        end = time.time()
        total_time = end-start
        print(f"Total training time {total_time:.2f} secs")

In [6]:
trainer = EmbeddingsClassifier(
                  model=LogisticRegression, 
                  train_dataset=(X_train_embeddings, y_train),
                  val_dataset=(X_val_embeddings, y_val),
                  directory="models",
                  params={"random_state":42, "max_iter":1000},
                  log_experiment=True,
                  experiment_name="doc_routing",
                  )

# train model
trainer.train()

Train Examples: 14192
Training and Logging to MLFLOW experiment doc_routing...
🏃 View run trusting-shrew-97 at: http://127.0.0.1:5000/#/experiments/1/runs/edfecf753754456981c488dd6e3aa6e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Total training time 7.18 secs


# **HELPER FUNCTIONS**

#### **Predictor class that loads the best checkpoint using a saved directory path**

In [7]:
class EmbeddingsPredictor:
    def __init__(self, model):       
        self.embeddings_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.model = model

    @classmethod
    def from_checkpoint(cls, checkpoint: str):
        artifacts = joblib.load(checkpoint)
        model = artifacts['model']
        return cls(model)

    def __call__(self, texts:list[str]):
        embeddings = self.embeddings_model.encode(texts)
        return self.model.predict(embeddings)

    def predict_embeddings_only(self, embeddings):
        return self.model.predict(embeddings)

In [18]:
import time

In [ ]:
# predict proba
def predict_with_proba_embeddings(text, predictor, class_to_idx):
    start_time = time.time()
    cleaned_text = clean_text_for_embeddings(text)
    dense_embeddings = predictor.embeddings_model.encode([cleaned_text])
    predicted_probas = predictor.model.predict_proba(dense_embeddings)
    end_time = time.time()
    result = []
    for scores in predicted_probas:
        pred = int(scores.argmax())
        all_probs = {label:round(float(score), 4) for label, score in zip(class_to_idx, scores)}
        sorted_probs = dict(sorted(all_probs.items(), key=lambda x: x[1], reverse=True))
        result.append({
            "prediction":class_to_idx[pred],
            "probabilities":sorted_probs,
            "latency":f"{(end_time-start_time)*1000:.2f} ms"
            })

    return result

In [21]:
checkpoint = Path().cwd().parent / "models/embeddings_model.joblib"
predictor = EmbeddingsPredictor.from_checkpoint(checkpoint)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [45]:
labels = list(class_to_idx)

In [47]:
pred_w_probs = predict_with_proba_embeddings("my house loan was not approved", predictor, class_to_idx=labels)
pred_w_probs

[{'prediction': 'Mortgage & Home Lending',
  'probabilities': {'Mortgage & Home Lending': 0.7194,
   'Consumer Lending': 0.273,
   'Credit Reporting & Disputes': 0.0025,
   'Cards & Payments': 0.0023,
   'Collections & Recovery': 0.002,
   'Banking Operations': 0.0006,
   'Money Transfer and Payments': 0.0002},
  'latency': '1123.20 ms'}]

In [42]:
dict(sorted(pred_w_probs[0]['probabilities'].items(), key=lambda x: x[1], reverse=True))

{'Banking Operations': 0.719,
 'Cards & Payments': 0.0834,
 'Money Transfer and Payments': 0.0763,
 'Credit Reporting & Disputes': 0.0608,
 'Collections & Recovery': 0.0441,
 'Consumer Lending': 0.0151,
 'Mortgage & Home Lending': 0.0015}

### **Evaluate model**

In [11]:
y_pred = predictor.predict_embeddings_only(X_val_embeddings)

accuracy = accuracy_score(y_true=y_val, y_pred=y_pred)
weighted_f1 = f1_score(y_true=y_val, y_pred=y_pred, average='weighted')
metrics = precision_recall_fscore_support(y_true=y_val, y_pred=y_pred, average='macro')

print(f"{classification_report(y_true=y_val, y_pred=y_pred)}\n")
metrics = {"accuracy": accuracy, 
               "precision_macro": metrics[0],
               "recall_macro": metrics[1],
               "f1_macro":metrics[2],
               "weighted_f1":weighted_f1
               }
print(json.dumps(metrics, indent=3))

              precision    recall  f1-score   support

           0       0.77      0.83      0.80       754
           1       0.79      0.82      0.80       754
           2       0.87      0.91      0.89       962
           3       0.82      0.74      0.78       423
           4       0.76      0.56      0.65       110
           5       0.76      0.62      0.68       280
           6       0.94      0.89      0.92       267

    accuracy                           0.82      3550
   macro avg       0.81      0.77      0.79      3550
weighted avg       0.82      0.82      0.82      3550


{
   "accuracy": 0.8180281690140845,
   "precision_macro": 0.8144720464419454,
   "recall_macro": 0.767892760615151,
   "f1_macro": 0.7875914661014098,
   "weighted_f1": 0.8161777776050775
}


In [48]:
import requests

In [74]:
text = """I am formally disputing the XXXX XXXX XXXX listed on my XXXX credit report. This item does not appear on my XXXX credit reports, which raises concerns regarding the accuracy and completeness of the information being reported by XXXX Additionally, this bankruptcy was dismissed and not discharged. Please verify that all information, including status and dates, is being reported correctly. Under the Fair Credit Reporting Act, I am requesting a detailed description of the method of verification used to confirm this public record. Please include the source of the information and how it was verified. If XXXX is unable to properly verify this item, I request that it be deleted from my credit file immediately. Sincerely, [ XXXX XXXX XXXX # XXXX"""
print(text)

I am formally disputing the XXXX XXXX XXXX listed on my XXXX credit report. This item does not appear on my XXXX credit reports, which raises concerns regarding the accuracy and completeness of the information being reported by XXXX Additionally, this bankruptcy was dismissed and not discharged. Please verify that all information, including status and dates, is being reported correctly. Under the Fair Credit Reporting Act, I am requesting a detailed description of the method of verification used to confirm this public record. Please include the source of the information and how it was verified. If XXXX is unable to properly verify this item, I request that it be deleted from my credit file immediately. Sincerely, [ XXXX XXXX XXXX # XXXX


In [ ]:
val_path = r"C:\Users\Lenovo\code\complaint-project\datasets\val.csv"
url = f"http://127.0.0.1:8000/evaluate/?dataset_location={val_path}"
requests.post(url=url).json()

{'results': {'time stamp': 'August 23, 2026 08:46:37 PM',
  'total time': '3.77 mins',
  'overall': {'accuracy': 0.8346478873239437,
   'precision_macro': 0.8336530848047446,
   'recall_macro': 0.7883438349834414,
   'f1_macro': 0.8081514744643693,
   'f1_weighted': 0.8336757538457182,
   'num_samples': 3550.0},
  'per_class': {'Mortgage & Home Lending': {'precision': 0.9739130434782609,
    'recall': 0.8389513108614233,
    'f1': 0.9014084507042254,
    'num_samples': 267.0},
   'Collections & Recovery': {'precision': 0.880641925777332,
    'recall': 0.9126819126819127,
    'f1': 0.8963757018887187,
    'num_samples': 962.0},
   'Cards & Payments': {'precision': 0.8246753246753247,
    'recall': 0.8421750663129973,
    'f1': 0.8333333333333334,
    'num_samples': 754.0},
   'Banking Operations': {'precision': 0.7802197802197802,
    'recall': 0.8474801061007957,
    'f1': 0.8124602670057216,
    'num_samples': 754.0},
   'Consumer Lending': {'precision': 0.8296296296296296,
    'recal

In [ ]:
params = {"dataset_location":r"C:\Users\Lenovo\code\complaint-project\datasets\val.csv"}
url = f"http://127.0.0.1:8000/evaluate/"
requests.post(url=url, params=params).json()